# 4.2 · 回归诊断 / Regression Diagnostics

> **课程定位 / Where this fits**
> **Part 4 第 2 课**。4.1 列了五大假设, 这一课**实操检验**：残差图、异方差、多重共线(VIF)、影响点。**这是把"模型跑出来了"升级到"模型可信吗"**——诊断是资深 DS 和调包侠的分水岭。
> Where 4.1 listed assumptions, this lesson tests them. Diagnostics separate "it ran" from "it's trustworthy".

> 💡 **面试相关 / Interview-relevant**
> - "怎么判断线性回归假设是否满足" ★★★★
> - "什么是异方差, 怎么办" ★★★★（稳健 SE）
> - "VIF 是什么, 多少算高" ★★★★（多重共线）
> - "高杠杆点 vs 异常点 vs 影响点" ★★★

---

## 学习目标 / Learning Objectives
1. 用**残差图四件套**诊断线性/同方差/正态/独立。
2. 检验并应对**异方差**（稳健标准误 / WLS）。
3. 用 **VIF** 量化多重共线性, 知道处置手段。
4. 区分**杠杆/异常/影响**三种问题点（Cook's distance）。

## 目录 / TOC
1. [诊断的总框架 ⭐](#1)
2. [🏠 数据 + 拟合](#2)
3. [残差图四件套 ⭐](#3)
4. [异方差: 检验 + 稳健 SE](#4)
5. [多重共线: VIF ⭐](#5)
6. [影响点: 杠杆 / Cook's distance](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 诊断的总框架 ⭐ / The Framework

诊断 = **逐条检验 4.1 的五大假设 + 找问题数据点**。

| 诊断什么 | 工具 | 违反 → 后果 |
|---|---|---|
| 线性 | 残差 vs 拟合值图 | 残差有曲率模式 → 系数有偏 |
| 同方差 | 残差散布是否均匀 + BP 检验 | 喇叭形 → SE 错, p 值/CI 不可信 |
| 残差正态 | QQ 图 (2.2) | 偏离对角 → 小样本 CI 不准 |
| 独立 | Durbin-Watson | 时序自相关 → SE 低估 |
| 无共线 | VIF | VIF 高 → 系数不稳/符号乱 |
| 问题点 | 杠杆 + Cook's distance | 个别点主导整条回归线 |

**核心心法**：残差里**不该有任何模式**。如果残差图能看出结构, 说明模型漏掉了信息。
The core mantra: residuals should contain no pattern. Visible structure means the model missed something.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import statsmodels.api as sm
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
# 取子样本加速, 用 statsmodels (要完整统计) / subsample + statsmodels
df = data.frame.sample(3000, random_state=0).reset_index(drop=True)
X = df[["MedInc","HouseAge","AveRooms","AveBedrms","Population","AveOccup"]]
y = df["MedHouseVal"]

X_sm = sm.add_constant(X)
model = sm.OLS(y, X_sm).fit()
print(f"R² = {model.rsquared:.3f}, adjusted R² = {model.rsquared_adj:.3f}")
print(f"\n💡 adjusted R²: R² 永随特征增多而升 (即使加噪声列), adj R² 惩罚特征数 → 选模型看它")


<a id="3"></a>
## 3. 残差图四件套 ⭐ / The Four Residual Plots

statsmodels 风格的四张诊断图——**一眼看出假设是否成立**。


In [ ]:
resid = model.resid
fitted = model.fittedvalues
std_resid = model.get_influence().resid_studentized_internal

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. 残差 vs 拟合值: 检验线性 + 同方差 / linearity + homoscedasticity
axes[0,0].scatter(fitted, resid, alpha=0.2, s=8)
axes[0,0].axhline(0, color="r", ls="--")
axes[0,0].set_xlabel("fitted"); axes[0,0].set_ylabel("residual")
axes[0,0].set_title("残差 vs 拟合: 应无模式无喇叭\n(这里有喇叭形→异方差!)")

# 2. QQ 图: 检验残差正态 / normality of residuals
st.probplot(std_resid, dist="norm", plot=axes[0,1])
axes[0,1].set_title("QQ 图: 应贴对角线 (2.2 节)")
axes[0,1].get_lines()[0].set(markersize=3, alpha=0.4)

# 3. Scale-Location: 检验同方差 (√|标准化残差| vs 拟合) / spread-location
axes[1,0].scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.2, s=8)
axes[1,0].set_xlabel("fitted"); axes[1,0].set_ylabel("√|std resid|")
axes[1,0].set_title("Scale-Location: 应水平 (上升→异方差)")

# 4. 残差直方图 / residual histogram
axes[1,1].hist(resid, bins=40); axes[1,1].set_xlabel("residual")
axes[1,1].set_title(f"残差分布 (skew={st.skew(resid):.2f})")
plt.tight_layout(); plt.show()
print("诊断结论: 残差 vs 拟合有喇叭形 + 右偏 → 违反同方差 + 正态. 房价数据典型问题")


<a id="4"></a>
## 4. 异方差: 检验 + 稳健 SE / Heteroscedasticity

**异方差** = 残差方差随拟合值变化（喇叭形）。后果：系数估计仍无偏, 但**标准误错 → p 值/CI 不可信**。

**正式检验**：Breusch-Pagan（$H_0$: 同方差）。
**修复**：
1. **稳健标准误**（HC, White's）—— 不改系数, 只修正 SE（最常用 ⭐）
2. **加权最小二乘 WLS** —— 给方差大的点小权重
3. **变换 y**（log）—— 常一举消除异方差（2.1/3.6 节）


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# BP 检验 / Breusch-Pagan test
bp = het_breuschpagan(resid, X_sm)
print(f"Breusch-Pagan 检验: LM 统计量={bp[0]:.1f}, p={bp[1]:.2e}")
print(f"→ p < 0.05, 拒绝同方差假设, 确认异方差\n")

# 稳健标准误 (HC3) / robust standard errors
robust = sm.OLS(y, X_sm).fit(cov_type="HC3")
print("普通 SE vs 稳健 SE (HC3) 对比 (前4个系数):")
comp = pd.DataFrame({"coef": model.params[:4].round(3),
                     "SE_normal": model.bse[:4].round(4),
                     "SE_robust": robust.bse[:4].round(4)})
print(comp)
print("\n系数不变, 但 SE 变了 → 稳健 SE 是异方差下做推断的正确做法 (2.11 pairs bootstrap 是另一条路)")


<a id="5"></a>
## 5. 多重共线: VIF ⭐ / Multicollinearity & VIF

**多重共线** = 特征之间高度相关。后果：系数**不稳定**（数据微变系数大跳）、**符号可能反直觉**、**单个系数不可解释**（但预测不受影响）。

**VIF（方差膨胀因子）**量化：把特征 $j$ 用其他特征回归, 取 $R_j^2$, 则
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
- VIF = 1：完全不相关
- **VIF > 5**：值得关注
- **VIF > 10**：严重共线 ⚠

**处置**：删冗余特征 / PCA 降维(0.7) / 用 Ridge(4.4, 专治共线) / 合并特征。


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).sort_values("VIF", ascending=False)
print("各特征 VIF:")
print(vif.round(2).to_string(index=False))
print("\nAveRooms 和 AveBedrms 共线 (房间多卧室也多) → VIF 高")
print("处置: 删一个, 或造比率特征 bedrooms/rooms (3.6), 或上 Ridge (4.4)")


<a id="6"></a>
## 6. 影响点: 杠杆 / Cook's distance / Influential Points

**三种问题点要分清**（2.1/3.3 节异常值的回归版）：

| 类型 | 定义 | 危害 |
|---|---|---|
| **异常点 outlier** | y 偏离预测很远（大残差）| 拉高 RMSE |
| **高杠杆点 leverage** | x 极端（远离 x 中心）| **有潜力**主导回归线 |
| **影响点 influential** | 异常 + 高杠杆 = 真正改变拟合 | **删了系数大变** ⚠ |

**Cook's distance** 综合度量"删掉这点对全体拟合的影响", $D_i > 4/n$ 常作警戒线。
Cook's distance measures how much deleting a point changes the whole fit.


In [ ]:
influence = model.get_influence()
leverage = influence.hat_matrix_diag
cooks = influence.cooks_distance[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 杠杆 vs 标准化残差 (影响点在右上/右下角) / leverage vs residual
axes[0].scatter(leverage, std_resid, alpha=0.3, s=10)
axes[0].axhline(0, color="gray", ls="--")
axes[0].set_xlabel("leverage (杠杆)"); axes[0].set_ylabel("std residual")
axes[0].set_title("杠杆 vs 残差\n右上/右下角 = 高杠杆+大残差 = 影响点")

# Cook's distance / Cook's distance stem
axes[1].stem(cooks, markerfmt=",")
axes[1].axhline(4/len(y), color="r", ls="--", label=f"警戒线 4/n={4/len(y):.4f}")
axes[1].set_xlabel("sample index"); axes[1].set_ylabel("Cook's D")
axes[1].set_title(f"Cook's distance ({(cooks > 4/len(y)).sum()} 个超线)")
axes[1].legend()
plt.tight_layout(); plt.show()

n_influential = (cooks > 4/len(y)).sum()
print(f"{n_influential} 个高影响点 ({n_influential/len(y):.1%})")
print("处置: 先查是不是数据错误(3.3); 真实点别删, 改用稳健回归(4.15)")


<a id="7"></a>
## 7. 小结 / Summary

```
诊断 = 检验 4.1 五假设 + 找问题点; 心法: 残差里不该有模式
残差图四件套: 残差vs拟合(线性+同方差) / QQ(正态) / Scale-Location / 直方图
异方差: BP 检验 → 稳健 SE(HC3, 不改系数只修 SE) / WLS / log(y)
多重共线: VIF; >5 关注 >10 严重 → 删/PCA/Ridge(4.4)
影响点: 异常(大残差) + 杠杆(x极端) = 影响点(Cook's D > 4/n) → 稳健回归(4.15)
adjusted R²: 惩罚特征数, 选模型用它不用 R²
```

### 💡 面试速查
1. **残差图无模式 = 假设 OK**; 喇叭形=异方差, 曲率=非线性
2. **异方差**: 系数无偏但 SE 错 → 稳健 SE (HC3)
3. **VIF > 10 严重共线** → Ridge 专治 (4.4)
4. **影响点 = 异常 + 高杠杆**; Cook's distance 度量
5. **adj R² > R²** 用于选模型 (R² 永增)

### 下一节
**4.3 多项式回归**——4.1/4.2 的线性假设若被残差曲率打破, 第一招就是多项式: 让线性模型拟合非线性。
